# 論文 docx → J-STAGE 全文 XML 変換（全国数学教育学会誌『数学教育学研究』）

**ツールの版: 0.30.0**（試用版。正式版は 1.0 から）

論文の Word ファイル（.docx）を、J-STAGE の全文 XML（JATS 1.1）に変換します。パソコンに何かをインストールする必要はありません。
下のセルの ▶ を押すと、ツールの準備（2 分ほど）のあと、その下に画面が開きます。画面に docx と PDF、書誌情報を入れて「変換する」を押します。
メニューの「表示」でサイドバーなどを閉じると広く使えます。手順ごとにセルを実行する従来版のノートブックは、説明書の「使い方」にあります。

説明書: [使い方・結果の見方・できないこと](https://github.com/thjpme/jstgxmlcng_JASME/blob/main/docs/USAGE.md)　[AI の API キーと手動モード](https://github.com/thjpme/jstgxmlcng_JASME/blob/main/docs/USAGE_AI.md)　ソースコード: https://github.com/thjpme/jstgxmlcng_JASME

In [ ]:
#@title 準備して画面を開く（2 分ほどかかります）
#@markdown **CONVERT_EMF**: Word の図形（EMF/WMF）の図を変換するために、LibreOffice を裏でインストールします（通常はチェックのまま。数分かかりますが、画面は待たずに使えます）。
CONVERT_EMF = True  #@param {type:"boolean"}
#@markdown **USE_DEV_VERSION**: 開発者向け。開発中の版を試すときだけチェックします（ブランチ名を尋ねます）。
USE_DEV_VERSION = False  #@param {type:"boolean"}

import shutil, subprocess, sys
# このランタイムで一度ツールを使っていると、古い版がメモリに残ったままになる（入れ直しても反映されない）
loaded_before = "jasme_jats" in sys.modules
BRANCH = "main"
if USE_DEV_VERSION:
    BRANCH = input("開発版のブランチ名を入力して Enter を押してください: ").strip()
    if not BRANCH:
        raise SystemExit("ブランチ名が入力されていません。開発版を使わないときは、チェックを外して実行してください。")
if loaded_before:
    print("ツールは入っているので、画面を開き直します（新しい版を入れるときは、メニューの「ランタイム」→「セッションを再起動」を"
          "選んでから、このセルを実行します）。")
else:
    # LibreOffice はインストールに数分かかるので裏で入れる（変換のときに終わるのを待つ）
    LO_LOG = "/content/libreoffice_install.log"
    if CONVERT_EMF and not (shutil.which("soffice") or shutil.which("libreoffice")):
        if globals().get("LO_PROC") is None or LO_PROC.poll() is not None:
            LO_PROC = subprocess.Popen("apt-get -qq update && (apt-get -qq install -y --no-install-recommends "
                                       "libreoffice-draw-nogui || apt-get -qq install -y libreoffice-draw)",
                                       shell=True, stdout=open(LO_LOG, "w"), stderr=subprocess.STDOUT)
    url = f"git+https://github.com/thjpme/jstgxmlcng_JASME.git@{BRANCH}"
    for args in (["jasme-jats[ai,images,colab,gui] @ " + url],                          # ツールと必要なライブラリ
                 ["--force-reinstall", "--no-deps", "jasme-jats @ " + url]):  # ツール本体は毎回最新に入れ直す
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-2000:])
            raise SystemExit("ツールのインストールに失敗しました。" + ("ブランチ名を確認してください。" if USE_DEV_VERSION else
                                                                      "しばらくしてからもう一度実行してください。"))
    if loaded_before:
        raise SystemExit("ツールを最新の版に入れ替えましたが、このランタイムには前の版が読み込まれたままです。"
                         "メニューの「ランタイム」→「セッションを再起動」（表示によっては「ランタイムを再起動」）を選んでから、"
                         "このセルをもう一度実行してください。")
    # 版の表示は別のプロセスで行う（ここで読み込むと、次にこのセルを実行したときに「前の版が残っている」扱いになるため）
    ver = subprocess.run([sys.executable, "-c", "from jasme_jats.pipeline import tool_version; print(tool_version())"],
                         capture_output=True, text=True).stdout.strip()
    if not ver:
        raise SystemExit("ツールを読み込めませんでした。メニューの「ランタイム」→「セッションを再起動」を選んでから、このセルをもう一度実行してください。")
    print("準備ができました。ツールの版:", ver)
    if BRANCH != "main":
        print("※ 開発版（ブランチ", BRANCH, "）を使っています。")
    if not ver.startswith("0.30.0"):
        # ノートブックとツールの版が違うと、後の手順がツールにない機能を呼んで止まる。正式版（main）のときは、ここで止めて開き直してもらう
        mismatch = "入ったツールの版（" + ver + "）が、このノートブックの版（0.30.0）と違います。"
        if BRANCH != "main":
            print("※ " + mismatch + "手順の説明や画面が合わないことがあります。")
        else:
            raise SystemExit(mismatch + "ノートブックを GitHub（https://github.com/thjpme/jstgxmlcng_JASME）の main から開き直して、このセルからやり直してください。"
                             "開発中の版のノートブックを試しているときは、「開発版を使う（USE_DEV_VERSION）」にチェックを入れて、"
                             "そのブランチ名を入力してください。")
    if globals().get("LO_PROC") is not None and LO_PROC.poll() is None:
        print("LibreOffice を裏でインストールしています（数分）。インストールは裏で続くので、画面を開きます"
              "（変換は、インストールが終わってから始まります）。")
from jasme_jats.app import launch
if globals().get("_app") is not None:
    _app.close()                      # 開き直し
_app = launch()